# Memory State Debugging

Debug agent decisions by examining memory state at any point in time.

**Problem**: Agent makes unexpected decisions based on corrupted/incorrect memories with no visibility into which memories influenced the behavior.

In [ ]:
import asyncio
from memoir.core.memory import ProllyTreeMemoryStoreManager
from memoir.classifier.intelligent import IntelligentClassifier
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-4", temperature=0)
classifier = IntelligentClassifier(llm=llm)
memory = ProllyTreeMemoryStoreManager(classifier=classifier)

In [ ]:
# Simulate agent learning user preferences
commit1 = await memory.store_memory(
    "User prefers Python for data analysis",
    metadata={"message": "Initial preference"}
)

commit2 = await memory.store_memory(
    "User likes coffee in the morning",
    metadata={"message": "Daily routine"}
)

commit3 = await memory.store_memory(
    "User actually prefers R for statistics work",
    metadata={"message": "Preference update"}
)

In [ ]:
# Simulate problematic memory corruption
corrupted_commit = await memory.store_memory(
    "User hates all programming languages",  # This is wrong!
    metadata={"message": "Bad memory entry"}
)

print(f"Problematic commit: {corrupted_commit[:8]}")

In [ ]:
# Agent decision point: What language to recommend?
current_memories = await memory.search_memories("programming language preferences")
print("Current memory state (corrupted):")
for mem in current_memories[:3]:
    print(f"- {mem.content}")

In [ ]:
# Debug: Time-travel to before corruption
print(f"\nDebugging: Memory state at commit {commit3[:8]}")
clean_memories = await memory.search_memories(
    "programming language preferences", 
    at_commit=commit3
)
for mem in clean_memories[:3]:
    print(f"- {mem.content}")

In [ ]:
# Debug: Examine taxonomy state at different points
current_taxonomy = await memory.get_taxonomy_paths()
print(f"\nCurrent taxonomy has {len(current_taxonomy)} paths")

# Show which path the corrupt memory was classified to
corrupt_results = await memory.search_memories("hates all programming")
if corrupt_results:
    print(f"Corrupt memory path: {corrupt_results[0].path}")

In [ ]:
# Fix: Revert to clean state
await memory.checkout(commit3)
print(f"\nReverted to clean commit: {commit3[:8]}")

# Verify fix
fixed_memories = await memory.search_memories("programming language preferences")
print("Fixed memory state:")
for mem in fixed_memories[:3]:
    print(f"- {mem.content}")

In [ ]:
# Add correct memory update
final_commit = await memory.store_memory(
    "User prefers R for statistical analysis, Python for general data work",
    metadata={"message": "Corrected preference"}
)

print(f"Clean timeline restored with commit: {final_commit[:8]}")